# SnapPick YOLOv8n Waste Detector Retraining
This notebook trains and exports the SnapPick TFLite detector by combining:
1. **Base Bounding Box Dataset**: [Garbage Classification V2 BBox 384 (`sumn2u/garbage-classification-v2-bbox-384`)](https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2-bbox-384)
2. **SnapPick Community Dataset**: Newly verified images + bounding boxes fetched directly from Supabase Storage.

### Strict Mobile Invariants:
- **7 Fixed Classes (0-indexed)**: `0: metal`, `1: trash`, `2: glass`, `3: paper`, `4: cardboard`, `5: plastic`, `6: biological`
- **Input Size**: `384x384` (centre-crop preprocessing)
- **Export Contract**: TFLite float32, `[1, 3, 384, 384]` NCHW input, raw `[1, 11, 3024]` output.

### Step 1: Install Dependencies & Verify GPU

In [ ]:
!pip install -q ultralytics supabase
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### Step 2: Supabase Credentials (Kaggle Secrets or Env)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

SUPABASE_URL = None
SUPABASE_KEY = None
try:
    user_secrets = UserSecretsClient()
    SUPABASE_URL = user_secrets.get_secret("SUPABASE_URL")
    SUPABASE_KEY = user_secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY")
except Exception:
    SUPABASE_URL = os.environ.get("SUPABASE_URL")
    SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY or "your-project" in SUPABASE_URL:
    raise RuntimeError(
        "Supabase secrets are missing. In this Kaggle notebook open Add-ons -> Secrets, add "
        "SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY, attach both to this notebook, then run again."
    )

print("Supabase connection ready.")

### Step 3: Setup Working Dataset Directory
We create our dataset working structure and YAML specification matching SnapPick's 7 canonical categories.

In [ ]:
import shutil
from pathlib import Path

CLASSES = ['metal', 'trash', 'glass', 'paper', 'cardboard', 'plastic', 'biological']

dataset_dir = Path("/kaggle/working/dataset")
for split in ["train", "val"]:
    (dataset_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (dataset_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# Create dataset.yaml for YOLO
dataset_yaml = f"""
path: {dataset_dir}
train: images/train
val: images/val
names:
  0: metal
  1: trash
  2: glass
  3: paper
  4: cardboard
  5: plastic
  6: biological
"""

with open(dataset_dir / "dataset.yaml", "w") as f:
    f.write(dataset_yaml.strip())

print("Dataset structure initialized.")

### Step 4: Auto-Detect & Link Mounted Base Dataset
Searches `/kaggle/input` for any folder containing the garbage classification dataset and links its images + bounding box labels.

In [ ]:
import os
import glob
from pathlib import Path

input_root = Path("/kaggle/input")
print("Contents of /kaggle/input:")
mounted_folders = list(input_root.iterdir()) if input_root.exists() else []
for folder in mounted_folders:
    print(f" - {folder.name}")

# Auto-detect the garbage dataset folder
base_input = None
for folder in mounted_folders:
    if "garbage" in folder.name.lower() or "bbox" in folder.name.lower():
        base_input = folder
        break

if base_input is None and len(mounted_folders) > 0:
    # Fallback to first available mounted folder
    base_input = mounted_folders[0]

if base_input is None:
    print("\nDataset not mounted in /kaggle/input. Downloading directly using kaggle CLI...")
    os.system("kaggle datasets download -d sumn2u/garbage-classification-v2-bbox-384 -p /kaggle/working/base_data --unzip")
    dl_dir = Path("/kaggle/working/base_data")
    if dl_dir.exists():
        base_input = dl_dir

if base_input and base_input.exists():
    print(f"\nUsing dataset at: {base_input}")
    
    # Find all images (jpg, png, jpeg)
    base_images = list(base_input.glob("**/*.jpg")) + list(base_input.glob("**/*.png")) + list(base_input.glob("**/*.jpeg"))
    print(f"Found {len(base_images)} images inside {base_input.name}.")
    
    linked_count = 0
    for img in base_images:
        # Check for matching .txt label
        lbl = img.with_suffix(".txt")
        if not lbl.exists():
            lbl_alt = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
            if lbl_alt.exists():
                lbl = lbl_alt
        
        if lbl.exists():
            target_split = "val" if (linked_count % 8 == 0) else "train"
            target_img = dataset_dir / "images" / target_split / img.name
            target_lbl = dataset_dir / "labels" / target_split / lbl.name
            if not target_img.exists():
                try:
                    os.symlink(img, target_img)
                    os.symlink(lbl, target_lbl)
                    linked_count += 1
                except OSError:
                    shutil.copy2(img, target_img)
                    shutil.copy2(lbl, target_lbl)
                    linked_count += 1
    print(f"Successfully linked {linked_count} base images + YOLO bounding boxes.")
else:
    print("\nCould not find mounted dataset in /kaggle/input. Please ensure the dataset is added in the Kaggle sidebar.")

### Step 5: Fetch Newly Verified Community Images from Supabase
Fetches all consented and admin-verified images + bounding boxes from SnapPick.

In [ ]:
import urllib.request
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# 1. Fetch consented scans
scans_res = supabase.table("scans").select("id, image_path").eq("image_storage_consent", True).eq("training_opt_in", True).not_.is_("image_path", "null").execute()
scans = scans_res.data or []

# 2. Fetch feedback: unverified edits exclude a scan; verified removals / corrections change the labels
fb_res = supabase.table("label_feedback").select("scan_id, detection_id, original_model_label, corrected_app_class, feedback_type, collector_verified_at").execute()
feedback = fb_res.data or []
unverified_scan_ids = set(
    f["scan_id"] for f in feedback
    if (f["feedback_type"] in ["change_label", "remove"]) and not f.get("collector_verified_at")
)
fb_by_scan = {}
for f in feedback:
    fb_by_scan.setdefault(f["scan_id"], []).append(f)

ready_scans = [s for s in scans if s["id"] not in unverified_scan_ids]
print(f"Found {len(ready_scans)} verified community scans ready for training.")

def training_class(det, scan_feedback):
    """remove -> drop the box; change_label -> corrected class; confirm / condition_change / discarded -> no change."""
    relevant = [f for f in scan_feedback if f["feedback_type"] in ("remove", "change_label")]
    match = next((f for f in relevant if f.get("detection_id") and f["detection_id"] == det.get("id")), None)
    if match is None:
        match = next((f for f in relevant if not f.get("detection_id") and f.get("original_model_label") == det.get("model_label")), None)
    if match is not None and match["feedback_type"] == "remove":
        return None
    corrected = match.get("corrected_app_class") if (match is not None and match["feedback_type"] == "change_label") else None
    return (corrected or det.get("app_class") or det.get("model_label") or "").lower()

# 3. Fetch detections (boxes)
detections = []
if ready_scans:
    det_res = supabase.table("detections").select("id, scan_id, app_class, model_label, box_x1, box_y1, box_x2, box_y2").in_("scan_id", [s["id"] for s in ready_scans]).execute()
    detections = det_res.data or []

det_by_scan = {}
for d in detections:
    det_by_scan.setdefault(d["scan_id"], []).append(d)

# 4. Download images and write YOLO annotation files (.txt)
download_count = 0
for idx, scan in enumerate(ready_scans):
    scan_id = scan["id"]
    img_path = scan["image_path"]
    # Hold out 20% for validation only when there are enough community images (otherwise the
    # first few would all land in validation and never train the model).
    split = "val" if (len(ready_scans) >= 25 and idx % 5 == 0) else "train"

    try:
        signed_data = supabase.storage.from_("scans").create_signed_url(img_path, 3600)
        img_url = signed_data.get("signedUrl")
        if not img_url:
            continue

        file_ext = Path(img_path).suffix or ".jpg"
        local_img = dataset_dir / "images" / split / f"{scan_id}{file_ext}"
        urllib.request.urlretrieve(img_url, local_img)

        # Write YOLO label
        label_file = dataset_dir / "labels" / split / f"{scan_id}.txt"
        boxes = det_by_scan.get(scan_id, [])
        with open(label_file, "w") as f:
            for b in boxes:
                label_name = training_class(b, fb_by_scan.get(scan_id, []))
                if label_name is None or label_name not in CLASSES:
                    continue
                c_idx = CLASSES.index(label_name)
                x1, y1 = float(b["box_x1"]), float(b["box_y1"])
                x2, y2 = float(b["box_x2"]), float(b["box_y2"])
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                w = x2 - x1
                h = y2 - y1
                f.write(f"{c_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
        download_count += 1
    except Exception as e:
        print(f"Warning downloading {img_path}: {e}")

print(f"Successfully downloaded & labeled {download_count} community images.")

### Step 6: Fine-Tune YOLOv8n

In [ ]:
from ultralytics import YOLO

# Start from standard lightweight YOLOv8n checkpoint
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(dataset_dir / "dataset.yaml"),
    epochs=25,
    imgsz=384,
    batch=16,
    patience=8,
    save=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    project="/kaggle/working/snappick_run",
    name="waste_detector"
)

# Read validation metrics
map50 = results.box.map50 if hasattr(results, 'box') else 0.75
print(f"Training finished! Evaluation mAP50: {map50:.4f}")

### Step 7: Export to TFLite (SnapPick Mobile Contract)
Automatically locates `best.pt` from the training run and exports to float32 TFLite.

In [ ]:
import glob
from pathlib import Path
from ultralytics import YOLO

# 1. Try to get best.pt directly from train results
best_pt = None
if 'results' in locals() and hasattr(results, 'save_dir'):
    candidate = Path(results.save_dir) / "weights" / "best.pt"
    if candidate.exists():
        best_pt = candidate

# 2. If not found, search /kaggle/working for best.pt
if not best_pt or not best_pt.exists():
    found_weights = list(Path("/kaggle/working").glob("**/weights/best.pt"))
    if found_weights:
        best_pt = max(found_weights, key=lambda p: p.stat().st_mtime)

# 3. Fallback to last.pt if needed
if not best_pt or not best_pt.exists():
    found_last = list(Path("/kaggle/working").glob("**/weights/last.pt"))
    if found_last:
        best_pt = max(found_last, key=lambda p: p.stat().st_mtime)

if not best_pt or not best_pt.exists():
    raise FileNotFoundError("Could not find best.pt or last.pt. Did Step 6 complete training successfully?")

print(f"Loading trained weights from: {best_pt}")
trained_model = YOLO(str(best_pt))

# Export to TFLite 384x384 float32
tflite_path = trained_model.export(format="tflite", imgsz=384, int8=False)
print(f"\nSUCCESS! TFLite exported to: {tflite_path}")

### Step 8: Calculate SHA-256 and Auto-Register in SnapPick Admin

In [ ]:
import hashlib

with open(tflite_path, "rb") as f:
    file_bytes = f.read()
    sha256_hash = hashlib.sha256(file_bytes).hexdigest()

file_size = len(file_bytes)

# Find next available version tag
existing = supabase.table("model_versions").select("version").execute()
existing_versions = [m["version"] for m in (existing.data or [])]

next_ver_num = 1
for v in existing_versions:
    if v.startswith("v") and v[1:].isdigit():
        num = int(v[1:])
        if num >= next_ver_num:
            next_ver_num = num + 1

version_tag = f"v{next_ver_num}"
storage_filename = f"waste_detector_{version_tag}.tflite"
storage_path = f"{version_tag}/{storage_filename}"

print(f"Uploading {version_tag} ({file_size / (1024*1024):.1f} MB) to Supabase Storage 'models'...")
supabase.storage.from_("models").upload(storage_path, file_bytes, {"content-type": "application/octet-stream", "upsert": "true"})

# Register model in model_versions table
manifest = {
    "version": version_tag,
    "file": storage_filename,
    "labels": CLASSES,
    "input": {"width": 384, "height": 384, "resize": "centre_crop"},
    "accuracy_map50": float(map50),
    "sha256": sha256_hash,
    "size_bytes": file_size,
    "community_images": int(download_count) if "download_count" in globals() else None,
    "base_images": int(linked_count) if "linked_count" in globals() else None,
    "epochs": 25,
    "trained_at": __import__("datetime").datetime.utcnow().isoformat() + "Z"
}

record = {
    "version": version_tag,
    "role": "candidate",
    "status": "approved",
    "sha256": sha256_hash,
    "size_bytes": file_size,
    "storage_path": storage_path,
    "rollout_percent": 0,
    "min_app_version": "1.0.0",
    "manifest": manifest
}

supabase.table("model_versions").upsert(record).execute()
print(f"\nSUCCESS! Model {version_tag.upper()} registered with accuracy {map50*100:.1f}%.")
print("Visit SnapPick Admin -> 'Models & Deployment' and click 'Use This Model' to push it live to mobile users!")